In [2]:
from dotenv import load_dotenv
from langchain_tavily import TavilySearch

from utils.std_model import base_model

# 加载环境变量 .env 文件
load_dotenv()

chatLLM = base_model()

In [8]:

from deepagents import create_deep_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_deep_agent(
    model=chatLLM,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='e1c01d90-30f6-4d9d-9db1-36b4020a97ce'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_132da64d4c5444abaf5672', 'type': 'function', 'function': {'name': 'get_weather', 'arguments': '{"city": "sf"}'}}]}, response_metadata={'model_name': 'qwen-plus', 'finish_reason': 'tool_calls', 'request_id': 'b3b15c36-3690-9636-a6b1-4cdee33fb2b9', 'token_usage': {'input_tokens': 5926, 'output_tokens': 19, 'total_tokens': 5945, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--019e0b93-823a-7e11-bda4-0bd895d84a0e-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'sf'}, 'id': 'call_132da64d4c5444abaf5672', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content="It's always sunny in sf!", name='get_weather', id='b666cf1f-bf81-4f3f-ab20-4bb5115136da', tool_call_id='call_132da64d4c5444abaf5672'),
  AIMessage(content="It's always 

In [17]:
from langchain_core.tracers import langchain, ConsoleCallbackHandler
from IPython.core.display import Markdown
from typing import Literal
from utils.std_tavily import tavily_client

# System prompt to steer the agent to be an expert researcher
research_instructions = """You are an expert researcher. Your job is to conduct thorough research and then write a polished report.

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

tavily_client = tavily_client()
langchain.debug = True

def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

agent = create_deep_agent(
    model=chatLLM,
    tools=[internet_search],
    system_prompt=research_instructions,
)

response = agent.invoke({"messages": [{"role": "user", "content": "What is langgraph?"}]},config={"callbacks": [ConsoleCallbackHandler()]})

Markdown(response['messages'][-1].content)

[chain/start] [chain:LangGraph] Entering Chain run with input:
{
  "messages": [
    {
      "role": "user",
      "content": "What is langgraph?"
    }
  ]
}
[chain/start] [chain:LangGraph > chain:PatchToolCallsMiddleware.before_agent] Entering Chain run with input:
[inputs]
[chain/end] [chain:LangGraph > chain:PatchToolCallsMiddleware.before_agent] s] Exiting Chain run with output:
[outputs]
[chain/start] [chain:LangGraph > chain:model] Entering Chain run with input:
[inputs]
[llm/start] [chain:LangGraph > chain:model > llm:ChatTongyi] Entering LLM run with input:
{
  "prompts": [
    "System: You are an expert researcher. Your job is to conduct thorough research and then write a polished report.\n\nYou have access to an internet search tool as your primary means of gathering information.\n\n## `internet_search`\n\nUse this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.\n\n\nYou 

LangGraph is an open-source framework developed by the LangChain team that enables developers to build, coordinate, and execute complex AI agent workflows using graph-based architectures.

Here's what makes LangGraph distinctive:

**Core Concept**: It models AI workflows as directed graphs where:
- **Nodes** represent individual agents, LLM calls, or processing steps
- **Edges** define the flow of data and control between these components

**Key Capabilities**:
- **State Management**: Maintains shared state across multiple steps and agents
- **Cyclical Workflows**: Supports loops and iterative processes (unlike linear chains)
- **Multi-Agent Coordination**: Enables collaboration between specialized agents
- **Conditional Branching**: Allows workflows to make decisions and follow different paths
- **Error Handling**: Built-in mechanisms for managing failures in complex workflows

**Relationship to LangChain**: LangGraph extends LangChain's capabilities. You can use LangChain components (prompt templates, document loaders, vector stores, etc.) as building blocks within LangGraph nodes, but LangGraph provides the orchestration layer for more sophisticated, stateful applications.

**Use Cases**: Ideal for production-grade applications like:
- Multi-agent systems (research agents, planning agents, execution agents)
- ReAct-style agents that reason, act, and observe iteratively
- Complex chatbots with memory and context management
- Workflow automation requiring decision points and loops

In essence, while LangChain excels at linear "chain" operations, LangGraph provides the graph-based structure needed for sophisticated, stateful, multi-step AI applications.